# Séance 7 · Projet Kaggle Titanic (1/2) : explorer, préparer, première soumission · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

C'est le grand projet du parcours. Le 15 avril 1912, le Titanic coule : 1 502 personnes sur 2 224 n'ont pas survécu. Kaggle nous donne la fiche de 891 passagers (âge, classe, prix du billet, famille à bord...) **et** s'ils ont survécu. Ton but : **prédire** la survie de 418 autres passagers dont on te cache la réponse, puis comparer ton score avec le monde entier.

Ce notebook tourne dans **Google Colab** (rien à installer). Exécute chaque cellule avec `Maj + Entrée`. Vous travaillez en **binôme** : un qui tape, un qui réfléchit à voix haute, et on échange les rôles toutes les 20 minutes.

**Livrable de la séance** : une première soumission sur Kaggle, avec le score noté dans la cellule prévue à la fin.


## Préparation

Tout est déjà installé sur Colab. On importe juste nos outils habituels.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
print("Prêt !")

## 1. Kaggle et la compétition Titanic

**Kaggle** est le terrain d'entraînement des data scientists du monde entier : des jeux de données, des notebooks partagés, et des **compétitions** avec un classement public. La compétition *Titanic* est le point d'entrée classique : plus d'un million de personnes y ont participé avant toi.

### S'inscrire et récupérer les données (5 min)
1. Va sur https://www.kaggle.com et crée un compte (gratuit, avec un e-mail ou un compte Google).
2. Ouvre https://www.kaggle.com/competitions/titanic → bouton **Join Competition** → **I Understand and Accept**.
3. Onglet **Data** → tout en bas, bouton **Download All** (un zip avec 3 fichiers).
4. Dans Colab, clique sur l'icône **dossier** dans la barre de gauche, puis glisse `train.csv` et `test.csv` dedans. Attention : les fichiers disparaissent quand la session Colab se ferme (à refaire à la séance 8).

### Les 3 fichiers
| Fichier | Contenu | Rôle |
|---|---|---|
| `train.csv` | 891 passagers **avec** la colonne `Survived` | pour apprendre |
| `test.csv` | 418 passagers **sans** `Survived` | pour prédire et soumettre |
| `gender_submission.csv` | un exemple de fichier de soumission | pour voir le format attendu |

### Soumettre (à la fin de la séance)
1. Sur la page de la compétition, bouton **Submit Prediction**.
2. Glisse ton fichier `submission.csv` (418 lignes, colonnes `PassengerId` et `Survived`).
3. Clique **Submit** : en quelques secondes, Kaggle affiche ton **score** = pourcentage de bonnes prédictions sur une partie des 418 passagers.
4. Tu as droit à 10 soumissions par jour. Le classement s'appelle le **Leaderboard**.

Si tu n'as pas encore les fichiers, la cellule suivante charge une copie publique de `train.csv` pour s'entraîner.

In [ ]:
URL_SECOURS = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if os.path.exists("train.csv"):
    train = pd.read_csv("train.csv")
    print("train.csv de Kaggle chargé")
else:
    try:
        train = pd.read_csv(URL_SECOURS)
        print("Copie publique chargée (mêmes colonnes que train.csv). Pour la vraie compétition, dépose train.csv dans Colab.")
    except Exception as erreur:
        print("Impossible de charger les données : pas de réseau ?", erreur)
        raise

# test.csv n'a pas la colonne Survived : c'est à nous de la prédire
test = pd.read_csv("test.csv") if os.path.exists("test.csv") else None
print("test.csv :", "chargé" if test is not None else "absent (tout fonctionne quand même, sauf la soumission)")
train.head()

**Exercice** : combien de lignes et de colonnes dans `train` ? Affiche aussi les 3 derniers passagers.

<details><summary>Solution</summary>

```python
print(train.shape)      # (891, 12) : 891 passagers, 12 colonnes
train.tail(3)
```
</details>

In [ ]:
# À toi

## 2. Explorer : le dictionnaire des colonnes

Avant de modéliser, un data scientist lit la **documentation** des données. Voici la traduction des 12 colonnes :

| Colonne | Signification | Type |
|---|---|---|
| `PassengerId` | numéro du passager | identifiant (inutile pour prédire) |
| `Survived` | 1 = a survécu, 0 = non | **la cible**, ce qu'on doit prédire |
| `Pclass` | classe du billet (1 = première, 3 = troisième) | nombre |
| `Name` | nom complet, avec le titre (Mr, Mrs, Miss, Master...) | texte |
| `Sex` | `male` / `female` | texte |
| `Age` | âge en années (manquant pour 177 passagers) | nombre |
| `SibSp` | frères, sœurs et conjoint à bord | nombre |
| `Parch` | parents et enfants à bord | nombre |
| `Ticket` | numéro du billet | texte |
| `Fare` | prix du billet (en livres de 1912) | nombre |
| `Cabin` | numéro de cabine (manquant pour 687 passagers) | texte |
| `Embarked` | port d'embarquement : C = Cherbourg, Q = Queenstown, S = Southampton | texte |

In [ ]:
train.info()          # type de chaque colonne et nombre de valeurs remplies

In [ ]:
print("Cases vides par colonne :")
print(train.isna().sum())
print()
print("Taux de survie global :", round(train["Survived"].mean() * 100, 1), "%")
train.describe().round(1)     # résumé des colonnes numériques

**Exercice** : avec `value_counts()`, compte les passagers par classe (`Pclass`) puis par port (`Embarked`). Combien de passagers en 3e classe ?

<details><summary>Solution</summary>

```python
print(train["Pclass"].value_counts())      # 491 passagers en 3e classe
print(train["Embarked"].value_counts())    # Southampton domine
```
</details>

In [ ]:
# À toi

## 3. Quatre hypothèses à vérifier

Une **hypothèse**, c'est une idée qu'on peut vérifier avec les données. Le data scientist travaille comme un détective : il ne devine pas, il vérifie. Voici 4 idées à tester avec `groupby` et un graphique.

### Hypothèse 1 : « les femmes et les enfants d'abord »
C'est la règle annoncée sur le Titanic. A-t-elle été appliquée ?

In [ ]:
print(train.groupby("Sex")["Survived"].mean().round(2))

# Enfant = moins de 12 ans (un âge manquant donne False : on ne sait pas)
train["Enfant"] = train["Age"] < 12
print()
print(train.groupby("Enfant")["Survived"].mean().round(2))

### Hypothèse 2 : la classe compte
Les cabines de 1re classe étaient sur les ponts supérieurs, près des canots.

In [ ]:
print(train.groupby("Pclass")["Survived"].mean().round(2))

train.groupby(["Pclass", "Sex"])["Survived"].mean().unstack().plot(kind="bar", figsize=(7, 4))
plt.title("Taux de survie par classe et par sexe")
plt.ylabel("Taux de survie")
plt.xticks(rotation=0)
plt.show()

### Hypothèse 3 : le prix du billet
Si la classe compte, le prix devrait compter aussi. On découpe les prix en 4 tranches de même taille avec `pd.qcut`.

In [ ]:
print("Prix médian des survivants / non-survivants :")
print(train.groupby("Survived")["Fare"].median().round(1))

train["Tranche_prix"] = pd.qcut(train["Fare"], 4, labels=["bon marché", "moyen", "cher", "très cher"])
train.groupby("Tranche_prix", observed=True)["Survived"].mean().plot(kind="bar", figsize=(6, 4), color="tab:orange")
plt.title("Taux de survie selon le prix du billet")
plt.ylabel("Taux de survie")
plt.xticks(rotation=0)
plt.show()

### Hypothèse 4 : la famille à bord
Seul, on est libre de courir vers un canot... ou personne ne vous aide. On additionne `SibSp` + `Parch` + 1 (soi-même).

In [ ]:
train["Famille"] = train["SibSp"] + train["Parch"] + 1

survie_famille = train.groupby("Famille")["Survived"].agg(["mean", "count"]).round(2)
print(survie_famille)

survie_famille["mean"].plot(kind="bar", figsize=(7, 4), color="tab:green")
plt.title("Taux de survie selon la taille de la famille")
plt.xlabel("Personnes de la famille à bord (soi-même inclus)")
plt.ylabel("Taux de survie")
plt.xticks(rotation=0)
plt.show()

### Bilan des hypothèses

| Hypothèse | Verdict | Ce que disent les chiffres |
|---|---|---|
| Femmes et enfants d'abord | ✅ | 74 % des femmes survivent contre 19 % des hommes ; les enfants s'en sortent mieux |
| La classe compte | ✅ | 63 % en 1re classe, 24 % en 3e |
| Le prix du billet | ✅ | les billets « très chers » survivent 2 fois plus que les « bon marché » |
| La famille | 🤔 | seul = 30 %, famille de 2 à 4 = plus de 50 %, famille de 5+ = ça chute |

Ces 4 hypothèses vont devenir les **variables** de notre modèle.

**Exercice** : hypothèse 5, le port d'embarquement. Le taux de survie change-t-il selon `Embarked` ? Trouve une explication (indice : regarde aussi la classe par port).

<details><summary>Solution</summary>

```python
print(train.groupby("Embarked")["Survived"].mean().round(2))
# Cherbourg (C) survit mieux... parce qu'il y avait plus de 1re classe :
print(pd.crosstab(train["Embarked"], train["Pclass"]))
```
</details>

In [ ]:
# À toi

## 4. Nettoyer : les cases vides

Un modèle refuse les cases vides. On a 3 colonnes à traiter :
- `Age` : 177 manquants (20 % !). Supprimer ces passagers = perdre un cinquième des données. On préfère **remplir** avec une valeur plausible : la **médiane** (la valeur du milieu, moins sensible aux extrêmes que la moyenne).
- `Embarked` : 2 manquants → on met le port le plus fréquent.
- `Fare` : rien de vide dans `train`, mais 1 manquant dans `test.csv` → médiane aussi.
- `Cabin` : 687 manquants, on laisse tomber cette colonne.

In [ ]:
print("Âge médian :", train["Age"].median(), "ans")
print("Port le plus fréquent :", train["Embarked"].mode()[0])
print("Prix médian :", round(train["Fare"].median(), 2), "livres")

# Avant / après remplissage de l'âge par la médiane
age_rempli = train["Age"].fillna(train["Age"].median())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
axes[0].hist(train["Age"].dropna(), bins=30, color="tab:blue")
axes[0].set_title("Âges connus (714 passagers)")
axes[1].hist(age_rempli, bins=30, color="tab:red")
axes[1].set_title("Après remplissage : un pic artificiel à 28 ans")
plt.show()

Le pic rouge est moche : 177 passagers ont soudain 28 ans. On fera mieux à la section 5 grâce au **titre** du passager.

**Exercice** : calcule l'âge médian par classe (`groupby("Pclass")`). Les passagers de 1re classe sont-ils plus vieux ?

<details><summary>Solution</summary>

```python
print(train.groupby("Pclass")["Age"].median())
# 1re classe : 37 ans, 3e classe : 24 ans. Remplir par classe serait déjà plus fin.
```
</details>

In [ ]:
# À toi

## 5. Créer de nouvelles variables

C'est l'étape préférée des data scientists : le **feature engineering**, ou l'art de donner de meilleurs indices au détective. On crée 3 variables :
1. `Famille` = taille de la famille à bord (déjà faite)
2. `Seul` = 1 si le passager voyage seul
3. `Titre` = Mr, Mrs, Miss, Master... extrait du nom

In [ ]:
train["Famille"] = train["SibSp"] + train["Parch"] + 1
train["Seul"] = (train["Famille"] == 1).astype(int)
print(train.groupby("Seul")["Survived"].mean().round(2))

### Extraire le titre avec une expression régulière
Un nom ressemble à `Braund, Mr. Owen Harris` : le titre est **entre la virgule et le point**. Une **expression régulière** (regex) est un motif de recherche dans du texte :
- `,\s*` : une virgule puis d'éventuels espaces
- `([^\.]+)` : tout ce qui n'est pas un point (c'est ce qu'on garde, entre parenthèses)
- `\.` : le point

In [ ]:
train["Titre"] = train["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip()
print(train["Titre"].value_counts())

17 titres différents, dont beaucoup rarissimes (Capt, Countess, Jonkheer...). Un modèle n'apprend rien d'un titre vu une seule fois : on **regroupe**.

In [ ]:
def regrouper_titre(titre):
    """Garde les 4 titres fréquents, regroupe le reste dans « Autre »."""
    if titre in ["Mr", "Mrs", "Miss", "Master"]:
        return titre
    if titre in ["Mlle", "Ms"]:
        return "Miss"
    if titre == "Mme":
        return "Mrs"
    return "Autre"          # Dr, Rev, Col, Major, Countess, Capt...


train["Titre"] = train["Titre"].apply(regrouper_titre)
print(train["Titre"].value_counts())

train.groupby("Titre")["Survived"].mean().sort_values().plot(kind="barh", figsize=(6, 3.5), color="tab:purple")
plt.title("Taux de survie par titre")
plt.xlabel("Taux de survie")
plt.show()

# Bonus : le titre donne aussi un bon âge pour les cases vides !
print(train.groupby("Titre")["Age"].median())

`Master` = petit garçon (en 1912, on appelait ainsi les garçons de moins de 13 ans). Le titre capture d'un coup le sexe, l'âge et le statut social : c'est l'une des variables les plus puissantes de cette compétition.

**Exercice** : crée la variable `Cabine_connue` (1 si `Cabin` n'est pas vide, sinon 0) et regarde le taux de survie. Pourquoi, à ton avis ?

<details><summary>Solution</summary>

```python
train["Cabine_connue"] = train["Cabin"].notna().astype(int)
print(train.groupby("Cabine_connue")["Survived"].mean().round(2))
# 67 % contre 30 % : les cabines connues sont surtout celles de 1re classe...
# et celles des survivants, qui ont pu raconter où ils dormaient.
```
</details>

In [ ]:
# À toi

## 6. La recette complète : `preparer()`

On range tout le nettoyage dans **une seule fonction**. Pourquoi ? Parce qu'il faudra appliquer **exactement la même recette** à `test.csv` avant de prédire. Une recette écrite une fois = zéro oubli.

In [ ]:
COLONNES = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Famille", "Seul", "Titre"]

def preparer(df):
    """Transforme le tableau brut de Kaggle en tableau de nombres prêt pour un modèle."""
    d = df.copy()
    # 1. Nouvelles variables
    d["Famille"] = d["SibSp"] + d["Parch"] + 1
    d["Seul"] = (d["Famille"] == 1).astype(int)
    d["Titre"] = d["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip().apply(regrouper_titre)
    # 2. Cases vides : l'âge médian de chaque titre (un « Master » a 4 ans, un « Mr » 30)
    d["Age"] = d.groupby("Titre")["Age"].transform(lambda s: s.fillna(s.median()))
    d["Age"] = d["Age"].fillna(d["Age"].median())
    d["Fare"] = d["Fare"].fillna(d["Fare"].median())
    d["Embarked"] = d["Embarked"].fillna("S")
    # 3. Tout en nombres
    d["Sex"] = (d["Sex"] == "female").astype(int)
    d["Embarked"] = d["Embarked"].map({"S": 0, "C": 1, "Q": 2})
    d["Titre"] = d["Titre"].map({"Mr": 0, "Mrs": 1, "Miss": 2, "Master": 3, "Autre": 4})
    return d[COLONNES]


X = preparer(train)          # ce que le modèle voit
y = train["Survived"]        # ce qu'il doit deviner
print(X.isna().sum().sum(), "case vide restante")
X.head()

**Exercice** : quelle variable est la plus liée à la survie ? Calcule `X.assign(Survived=y).corr()["Survived"]` et trie. Un nombre proche de 1 ou -1 = lien fort.

<details><summary>Solution</summary>

```python
correlations = X.assign(Survived=y).corr()["Survived"].drop("Survived").sort_values()
print(correlations.round(2))
correlations.plot(kind="barh", title="Corrélation avec la survie")
plt.show()
# Sex (0,54) et Titre en tête, Pclass négatif (plus la classe est grande, moins on survit)
```
</details>

In [ ]:
# À toi

## 7. Premier modèle : régression logistique, puis forêt

Deux modèles que tu as croisés à la séance 6 :
- La **régression logistique** : une balance qui pèse chaque indice (femme : +2 points, 3e classe : -1 point...) et bascule vers « survit » ou « ne survit pas ».
- La **forêt aléatoire** : 200 arbres de décision, chacun entraîné sur un échantillon un peu différent, qui **votent**.

Pour mesurer honnêtement, on utilise la **validation croisée** (`cross_val_score`) : on coupe les 891 passagers en 5 paquets, on apprend sur 4 et on teste sur le 5e, 5 fois de suite avec un paquet différent. C'est comme passer 5 contrôles blancs avec des sujets différents plutôt qu'un seul : la moyenne est plus fiable.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Un premier essai simple : on cache 25 % des passagers
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

logistique = LogisticRegression(max_iter=1000)
logistique.fit(X_train, y_train)
print("Régression logistique, précision sur les 25 % cachés :", round(logistique.score(X_test, y_test) * 100, 1), "%")

# Regardons 5 prédictions face à la réalité
apercu = X_test.head().copy()
apercu["Réel"] = y_test.head().values
print(apercu)

# Validation croisée : 5 contrôles blancs pour chaque modèle
foret = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)

for nom, modele in [("Régression logistique", logistique), ("Forêt aléatoire", foret)]:
    scores = cross_val_score(modele, X, y, cv=5)
    print(f"{nom:22s} : {scores.mean()*100:.1f} % (± {scores.std()*100:.1f})")

Autour de 80-83 % : c'est déjà un très bon niveau pour cette compétition.

**Exercice** : change `max_depth` de la forêt (3, 5, 10, `None` = sans limite). Note la précision à chaque fois. Que remarques-tu quand l'arbre n'a plus de limite ?

<details><summary>Solution</summary>

```python
for profondeur in [3, 5, 10, None]:
    f = RandomForestClassifier(n_estimators=200, max_depth=profondeur, random_state=42)
    print(profondeur, round(cross_val_score(f, X, y, cv=5).mean() * 100, 1), "%")
# Sans limite, le score baisse un peu : les arbres apprennent par cœur (on en reparle à la séance 8)
```
</details>

In [ ]:
# À toi

## 8. Projet : ta première soumission (80 min)

En binôme, dans l'ordre :
- **Étape A (20 min)** : refaites les sections 3 à 7 ensemble en commentant chaque résultat. Changez les rôles à mi-parcours.
- **Étape B (30 min)** : Questions 1 et 2 ci-dessous.
- **Étape C (20 min)** : générez `submission.csv`, soumettez sur Kaggle, notez votre score (Question 3).
- **Étape D (10 min)** : préparez ce que vous allez dire aux autres (voir « Pour montrer aux autres »).

In [ ]:
# Question 1 : ajoutez UNE variable de votre choix dans preparer() et remesurez la forêt.
# Idées : Enfant (Age < 12), Cabine_connue, Prix_par_personne = Fare / Famille
# Astuce : copiez la fonction preparer() ici, modifiez-la, et n'oubliez pas d'ajouter la colonne dans COLONNES.

In [ ]:
# Question 2 : auriez-vous survécu ? Remplissez votre fiche de passager et demandez au modèle.
foret.fit(X, y)

moi = pd.DataFrame([{
    "Pclass": 3,           # 1, 2 ou 3
    "Sex": 0,              # 1 = femme, 0 = homme
    "Age": 15,
    "Fare": 8.0,           # prix moyen d'un billet de 3e classe
    "Embarked": 1,         # 0 = Southampton, 1 = Cherbourg, 2 = Queenstown
    "Famille": 3,          # toi + les tiens
    "Seul": 0,
    "Titre": 0,            # 0 = Mr, 1 = Mrs, 2 = Miss, 3 = Master, 4 = Autre
}])

proba = foret.predict_proba(moi)[0][1]
print(f"Chance de survie estimée : {proba*100:.0f} %")
print("Verdict :", "tu survis" if proba > 0.5 else "tu n'as pas de chance...")

### Générer `submission.csv`

On entraîne le meilleur modèle sur **tous** les passagers de `train` (plus de données = meilleur modèle), on applique `preparer()` à `test`, et on écrit un fichier avec 2 colonnes : `PassengerId` et `Survived`.

In [ ]:
meilleur = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
meilleur.fit(X, y)          # on apprend sur les 891 passagers

if test is not None:
    predictions = meilleur.predict(preparer(test))
    submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": predictions})
    submission.to_csv("submission.csv", index=False)
    print("submission.csv créé :", len(submission), "lignes,", submission["Survived"].mean().round(2) * 100, "% de survivants prédits")
    print(submission.head())
else:
    print("test.csv absent : dépose-le dans Colab (panneau Fichiers à gauche) puis relance cette cellule.")

### Soumettre sur Kaggle (étape C)
1. Panneau **Fichiers** de Colab (icône dossier) → clic droit sur `submission.csv` → **Télécharger**.
2. Page de la compétition → **Submit Prediction** → glisse le fichier → **Submit**.
3. Attends quelques secondes : ton score s'affiche (par exemple `0.77511` = 77,5 % de bonnes prédictions).
4. Onglet **Leaderboard** : cherche ton nom. Tu es classé parmi des dizaines de milliers de participants !

In [ ]:
# Question 3 : notez ici le score affiché par Kaggle (ex. 0.77511)
SCORE_PREMIERE_SOUMISSION = None
MODELE_UTILISE = "Forêt aléatoire, 200 arbres, profondeur 5"
VARIABLES_UTILISEES = COLONNES

if SCORE_PREMIERE_SOUMISSION is None:
    print("Pas encore de score ? Soumets sur Kaggle puis remplace None par ton score.")
else:
    print(f"Première soumission : {SCORE_PREMIERE_SOUMISSION:.5f} avec {MODELE_UTILISE}")
    print("Variables :", ", ".join(VARIABLES_UTILISEES))

## À retenir

- **Kaggle** : des données, un classement public, et le même défi pour tout le monde. `train.csv` pour apprendre, `test.csv` pour prédire.
- **Explorer d'abord** : lire le dictionnaire des colonnes, compter les cases vides, vérifier ses hypothèses avec `groupby` et un graphique.
- Sur le Titanic : le sexe, la classe, le prix et la taille de la famille comptent. Les données confirment « les femmes et les enfants d'abord ».
- **Nettoyer** : remplir les cases vides (médiane), jamais les laisser.
- **Feature engineering** : `Famille`, `Seul`, `Titre` (extrait avec une regex) donnent de meilleurs indices au modèle.
- Une fonction `preparer()` = la même recette pour `train` et `test`.
- **Validation croisée** : 5 contrôles blancs valent mieux qu'un seul pour mesurer un modèle.
- Une soumission = un fichier `PassengerId, Survived`. Un score de 77-80 % est très bon pour un premier essai.

## Pour montrer aux autres

Chaque binôme a 2 minutes. Trois questions guides :
1. Quelle hypothèse vous a le plus surpris (ou confirmé) ?
2. Quelle variable avez-vous ajoutée à la Question 1, et le score a-t-il bougé ?
3. Votre score Kaggle, et une idée pour l'améliorer la prochaine fois.

## Liens
- La compétition : https://www.kaggle.com/competitions/titanic
- Le dictionnaire officiel des données : https://www.kaggle.com/competitions/titanic/data
- Tester une regex en direct : https://regex101.com
- La version spatiale, plus ludique : https://www.kaggle.com/competitions/spaceship-titanic